# Hands-On LangChain #7: Chat with Your Files

Companion to [the complete article](https://todatabeyond.com/blog/hands-on-langchain-for-llms-app-chat-with-your-files).

The default path generates its own PDFs and runs real MiniLM embeddings, Chroma, and FLAN-T5-small CPU inference without API keys. First-run model downloads require internet. These short PDFs are generated fixtures, NOT the original CS229 lectures. Tests verify history enters question rewriting, source retrieval, external GUI history, PDF upload isolation, reset behavior, and dashboard construction. They do not assert perfect question rewrites or answer quality. The small model can give incomplete answers. Legacy chains/memory remain in langchain-classic to preserve the source tutorial; new production applications should use explicit managed conversational state. Never upload private documents to a hosted model without permission.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-chroma>=1.0,<2" "langchain-classic>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-openai>=1.0,<2" "langchain-text-splitters>=1.0,<2" "transformers>=4.57,<5" "panel>=1.8,<2" "param>=2,<3" torch sentencepiece pypdf reportlab numpy python-dotenv

## 2. Real pretrained local models

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

class LocalMiniLMEmbeddings(Embeddings):
    def __init__(self):
        self.model = ONNXMiniLM_L6_V2(preferred_providers=["CPUExecutionProvider"])
        self.model.DOWNLOAD_PATH = Path(tempfile.gettempdir()) / "tdb-minilm-model-cache"

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [vector.tolist() for vector in self.model(texts)]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

embedding = LocalMiniLMEmbeddings()
print(f"Embedding dimensions: {len(embedding.embed_query('example'))}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM

torch.set_num_threads(2)
cache = str(Path(tempfile.gettempdir()) / "tdb-flan-t5-cache")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model.eval()
calls = []

class LocalAnswerLLM(LLM):
    @property
    def _llm_type(self): return "local-flan-t5-small"
    def get_num_tokens(self, text): return len(tokenizer.encode(text))
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
        if inputs["input_ids"].shape[1] > 512:
            raise ValueError("Prompt exceeds this demo's 512-token input budget.")
        with torch.inference_mode():
            tokens = answer_model.generate(**inputs, max_new_tokens=64, do_sample=False)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True)
        calls.append({"prompt": prompt, "answer": answer})
        return answer

llm = LocalAnswerLLM()
print("Models: all-MiniLM-L6-v2 + google/flan-t5-small (CPU, no API key)")

## 3. Create reproducible text-based PDFs

In [ ]:
from reportlab.pdfgen import canvas
from pathlib import Path
import tempfile

sample_folder = Path(tempfile.mkdtemp(prefix="tdb-chat-pdfs-"))
sample_pdf = sample_folder / "generated-lecture.pdf"
replacement_pdf = sample_folder / "replacement.pdf"

def write_pdf(path, pages):
    pdf = canvas.Canvas(str(path))
    for lines in pages:
        y = 760
        for line in lines:
            pdf.drawString(40, y, line)
            y -= 22
        pdf.showPage()
    pdf.save()

write_pdf(sample_pdf, [
    ["The teaching assistants are Maya Chen and Omar Ali.", "Their majors are not mentioned in these notes."],
    ["Probability and statistics are prerequisites for the machine learning class.", "These prerequisites help students understand uncertainty and evaluate models."],
    ["The class teaches supervised and unsupervised machine learning."],
    ["Linear algebra is a prerequisite because learning algorithms use vectors and matrices."],
])
write_pdf(replacement_pdf, [
    ["The astronomy class studies planets and stars."],
    ["The astronomy teaching assistant is Noor."],
    ["The astronomy class meets on Monday."],
    ["The observatory telescope is used for astronomy projects."],
])
print("Generated sample PDFs: 4 lecture pages + 4 replacement pages")

## 4. PDF loading, splitting, embeddings, and conversational retrieval
The same function appears in the article. Injecting the local models avoids paid calls; omitting them enables hosted defaults.

In [ ]:
import os
from uuid import uuid4
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

def load_db(file, chain_type="stuff", k=4, embedding_model=None, chat_model=None):
    # Inject local models for the companion; defaults make hosted calls.
    if chain_type != "stuff":
        raise ValueError("This dashboard demo supports the stuff chain.")
    if k < 1:
        raise ValueError("k must be positive.")
    documents = PyPDFLoader(str(file)).load()
    docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(documents)
    if not docs:
        raise ValueError("The PDF has no extractable text; scanned PDFs need OCR first.")
    embeddings = embedding_model or OpenAIEmbeddings(model="text-embedding-3-small")
    model = chat_model or ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"], temperature=0)
    db = Chroma.from_documents(docs, embeddings, collection_name="chat_" + uuid4().hex)
    retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": k})
    condense = PromptTemplate.from_template(
        "Rewrite the follow-up as a standalone question using the chat history. "
        "Do not answer it.\nHistory: {chat_history}\nFollow-up: {question}\nStandalone question:"
    )
    answer = PromptTemplate.from_template(
        "Answer using only this context. If the answer is absent, say you do not know."
        "\nContext: {context}\nQuestion: {question}\nAnswer:"
    )
    # Memory is managed externally by the GUI.
    return ConversationalRetrievalChain.from_llm(
        llm=model, chain_type=chain_type, retriever=retriever,
        condense_question_prompt=condense,
        combine_docs_chain_kwargs={"prompt": answer},
        return_source_documents=True, return_generated_question=True,
    )

In [ ]:
def local_load_db(file, chain_type="stuff", k=4):
    return load_db(file, chain_type, k, embedding_model=embedding, chat_model=llm)

qa = local_load_db(sample_pdf)
assert qa.retriever.invoke("teaching assistants")
calls.clear()
first = qa.invoke({"question": "Who are the teaching assistants?", "chat_history": []})
assert len(calls) == 1 and len(first["source_documents"]) == 4
history = [("Who are the teaching assistants?", first["answer"])]
follow = qa.invoke({"question": "What are their majors?", "chat_history": history})
assert len(calls) == 3
assert history[0][0] in calls[1]["prompt"] and history[0][1] in calls[1]["prompt"]
assert calls[1]["answer"] == follow["generated_question"]
assert follow["generated_question"] in calls[2]["prompt"]
print("First answer:", first["answer"])
print("Generated follow-up query:", follow["generated_question"])
print("Follow-up answer:", follow["answer"])
print("Conversation model calls: 1 initial + 2 follow-up")
print("Retrieved source passages:", len(follow["source_documents"]))

## 5. Internal buffer memory and explicit clearing
Memory needs an explicit answer output key when a chain also returns source documents and the rewritten query. Memory is state, not evidence of accurate reasoning; bound or summarize it in real applications.

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history", input_key="question", output_key="answer", return_messages=True)
memory_qa = local_load_db(sample_pdf)
memory_qa.memory = memory
calls.clear()
one = memory_qa.invoke({"question": "Who are the teaching assistants?"})
two = memory_qa.invoke({"question": "What are their majors?"})
assert len(memory.chat_memory.messages) == 4
assert "Who are the teaching assistants?" in calls[1]["prompt"]
memory.clear()
assert not memory.chat_memory.messages
print("Buffer memory: 4 messages after two turns; 0 after clear")

## 6. Runnable Panel dashboard
This is the complete updated GUI from the original tutorial, with local model injection for the default path. Run the final display cell in a Panel-capable Jupyter notebook. Terminal/script users can serve `dashboard` with `pn.serve(dashboard)` on localhost. GUI callback tests below are headless, not a browser screenshot or production security audit.

In [ ]:
import panel as pn
import param
pn.extension()

In [ ]:
import tempfile
from pathlib import Path
import panel as pn
import param

class cbfs(param.Parameterized):
    chat_history = param.List(default=[])
    answer = param.String(default="")
    db_query = param.String(default="")
    db_response = param.List(default=[])

    def __init__(self, default_pdf="docs/cs229_lectures/MachineLearning-Lecture01.pdf", loader=load_db, **params):
        super().__init__(**params)
        self.panels = []
        self.loader = loader
        self.loaded_file = str(default_pdf)
        self.upload_directory = tempfile.TemporaryDirectory(prefix="tdb-chat-upload-")
        self.qa = loader(self.loaded_file, "stuff", 4)

    def call_load_db(self, count):
        if count == 0 or not file_input.value:
            return pn.pane.Str(f"Loaded File: {self.loaded_file}")
        # Per-instance storage avoids overwriting another session's temp.pdf.
        upload_path = Path(self.upload_directory.name) / "uploaded.pdf"
        file_input.save(str(upload_path))
        button_load.loading = True
        try:
            new_qa = self.loader(str(upload_path), "stuff", 4)
        finally:
            button_load.loading = False
        self.qa = new_qa
        self.loaded_file = Path(file_input.filename or "uploaded.pdf").name
        self.clr_history()
        return pn.pane.Str(f"Loaded File: {self.loaded_file}")

    def convchain(self, query):
        if not query:
            return pn.WidgetBox(*self.panels, scroll=True)
        result = self.qa.invoke({"question": query, "chat_history": self.chat_history})
        # Assign a new list so Param detects the history change.
        self.chat_history = [*self.chat_history, (query, result["answer"])]
        self.db_query = result["generated_question"]
        self.db_response = result["source_documents"]
        self.answer = result["answer"]
        self.panels.extend([
            pn.Row("User:", pn.pane.Str(query, width=600)),
            pn.Row("ChatBot:", pn.pane.Str(self.answer, width=600, styles={"background-color": "#F6F6F6"})),
        ])
        inp.value = ""
        return pn.WidgetBox(*self.panels, scroll=True)

    @param.depends("db_query")
    def get_lquest(self):
        return pn.Column(pn.pane.Markdown("**Last question to DB:**"), pn.pane.Str(self.db_query or "No DB accesses so far"))

    @param.depends("db_response")
    def get_sources(self):
        return pn.WidgetBox(pn.pane.Markdown("**Result of DB lookup:**"), *[pn.pane.Str(doc) for doc in self.db_response], width=600, scroll=True)

    @param.depends("chat_history")
    def get_chats(self):
        return pn.WidgetBox(pn.pane.Markdown("**Current Chat History:**"), *[pn.pane.Str(exchange) for exchange in self.chat_history], width=600, scroll=True)

    def clr_history(self, event=None):
        self.chat_history = []
        self.panels = []
        self.answer = ""
        self.db_query = ""
        self.db_response = []
        inp.value = ""
        conversation_view.objects = []

cb = cbfs(default_pdf=sample_pdf, loader=local_load_db)
file_input = pn.widgets.FileInput(accept=".pdf")
button_load = pn.widgets.Button(name="Load DB", button_type="primary")
button_clearhistory = pn.widgets.Button(name="Clear History", button_type="warning")
button_clearhistory.on_click(cb.clr_history)
inp = pn.widgets.TextInput(placeholder="Enter text here…")
conversation_view = pn.Column(height=300, scroll=True)

def submit_query(event):
    if event.new:
        conversation_view.objects = [cb.convchain(event.new)]

inp.param.watch(submit_query, "value")
bound_button_load = pn.bind(cb.call_load_db, button_load.param.clicks)
tab1 = pn.Column(inp, pn.layout.Divider(), conversation_view)
tab2 = pn.Column(pn.panel(cb.get_lquest), pn.layout.Divider(), pn.panel(cb.get_sources))
tab3 = pn.Column(pn.panel(cb.get_chats))
tab4 = pn.Column(
    pn.Row(file_input, button_load, bound_button_load),
    button_clearhistory,
    pn.pane.Markdown("Upload a text-based PDF. Clearing history starts a new topic. Documents are sent to the configured model in hosted mode."),
)
dashboard = pn.Column(
    pn.pane.Markdown("# ChatWithYourData_Bot"),
    pn.Tabs(("Conversation", tab1), ("Database", tab2), ("Chat History", tab3), ("Configure", tab4)),
)
dashboard

## 7. Verify GUI conversation, reset, upload, and isolation

In [ ]:
calls.clear()
inp.value = "Who are the teaching assistants?"
assert len(cb.chat_history) == 1 and cb.db_response and cb.panels
inp.value = "What are their majors?"
assert len(cb.chat_history) == 2 and len(cb.panels) == 4
assert "Who are the teaching assistants?" in calls[1]["prompt"]
assert cb.db_query == calls[1]["answer"]
root = dashboard.get_root()
assert root is not None
cb.clr_history()
assert not cb.chat_history and not cb.panels and not cb.db_response
assert not cb.answer and not cb.db_query and not conversation_view.objects
print("Panel conversation: 2 turns, sources and rewritten query recorded")
print("Clear History: messages, answer, sources, query, and visible conversation reset")

file_input.value = replacement_pdf.read_bytes()
file_input.filename = "replacement.pdf"
cb.call_load_db(1)
assert cb.loaded_file == "replacement.pdf" and not cb.chat_history
new_docs = cb.qa.retriever.invoke("astronomy")
assert new_docs and all("astronomy" in d.page_content.lower() for d in new_docs)
assert all("Maya" not in d.page_content for d in new_docs)
other = cbfs(default_pdf=sample_pdf, loader=local_load_db)
assert other.upload_directory.name != cb.upload_directory.name
assert other.chat_history == []
other.upload_directory.cleanup()
print("PDF upload: replacement document indexed, prior conversation cleared")
print("Upload storage: distinct temporary directory per dashboard instance")
print("Panel dashboard: Bokeh root constructed successfully")
print("All local chat-with-files integration assertions passed.")

## 8. Display the dashboard

In [ ]:
dashboard

## 9. Complete opt-in hosted workflow

All eleven original code steps follow with maintained imports, defined model selection, fresh indexing, `.invoke`, explicit memory keys, and corrected Panel callbacks. Chroma replaces the original DocArray backend consistently with the series. The missing decorative `img/convchain.jpg` dependency is removed. Set your local `OPENAI_API_KEY` and `OPENAI_CHAT_MODEL`, provide CS229 lecture PDFs, and enable the flag. Hosted calls incur charges and send document text to the provider. The appendix was syntax-checked but not executed against hosted APIs; historical blog outputs are not reproducibility promises.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from pathlib import Path
    from tempfile import mkdtemp
    import panel as pn
    from dotenv import load_dotenv
    pn.extension()
    load_dotenv()
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import ChatOpenAI
    llm_name = os.environ["OPENAI_CHAT_MODEL"]
    llm = ChatOpenAI(model=llm_name, temperature=0)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # Build a fresh index; do not mix embeddings from different models.
    pdf_paths = [f"docs/cs229_lectures/MachineLearning-Lecture{i:02d}.pdf" for i in (1, 1, 2, 3)]
    if not all(Path(path).is_file() for path in pdf_paths):
        raise FileNotFoundError("Place the three CS229 PDFs under docs/cs229_lectures/.")
    documents = []
    for path in pdf_paths:
        documents.extend(PyPDFLoader(path).load())
    splits = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150).split_documents(documents)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")
    persist_directory = mkdtemp(prefix="langchain-chat-files-")
    vectordb = Chroma.from_documents(splits, embedding=embedding, collection_name="lectures", persist_directory=persist_directory)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "What are major topics for this class?"
    docs = vectordb.similarity_search(question,k=3)
    print(docs[0])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # Build prompt
    from langchain_core.prompts import PromptTemplate
    template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer. 
    {context}
    Question: {question}
    Helpful Answer:"""
    QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"],template=template,)
    
    # Run chain
    from langchain_classic.chains import RetrievalQA
    question = "Is probability a class topic?"
    qa_chain = RetrievalQA.from_chain_type(llm,
                                           retriever=vectordb.as_retriever(),
                                           return_source_documents=True,
                                           chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})
    
    
    result = qa_chain.invoke({"query": question})
    print(result["result"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.memory import ConversationBufferMemory
    
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        input_key="question",
        output_key="answer"
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.chains import ConversationalRetrievalChain
    retriever=vectordb.as_retriever()
    qa = ConversationalRetrievalChain.from_llm(
        llm,
        retriever=retriever,
        memory=memory
    )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "Is probability a class topic?"
    result = qa.invoke({"question": question})
    print(result['answer'])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "why are those prerequesites needed?"
    result = qa.invoke({"question": question})
    print(result["answer"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from uuid import uuid4
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_chroma import Chroma
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_classic.chains import ConversationalRetrievalChain
    from langchain_core.prompts import PromptTemplate
    
    def load_db(file, chain_type="stuff", k=4, embedding_model=None, chat_model=None):
        # Inject local models for the companion; defaults make hosted calls.
        if chain_type != "stuff":
            raise ValueError("This dashboard demo supports the stuff chain.")
        if k < 1:
            raise ValueError("k must be positive.")
        documents = PyPDFLoader(str(file)).load()
        docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(documents)
        if not docs:
            raise ValueError("The PDF has no extractable text; scanned PDFs need OCR first.")
        embeddings = embedding_model or OpenAIEmbeddings(model="text-embedding-3-small")
        model = chat_model or ChatOpenAI(model=os.environ["OPENAI_CHAT_MODEL"], temperature=0)
        db = Chroma.from_documents(docs, embeddings, collection_name="chat_" + uuid4().hex)
        retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": k})
        condense = PromptTemplate.from_template(
            "Rewrite the follow-up as a standalone question using the chat history. "
            "Do not answer it.\nHistory: {chat_history}\nFollow-up: {question}\nStandalone question:"
        )
        answer = PromptTemplate.from_template(
            "Answer using only this context. If the answer is absent, say you do not know."
            "\nContext: {context}\nQuestion: {question}\nAnswer:"
        )
        # Memory is managed externally by the GUI.
        return ConversationalRetrievalChain.from_llm(
            llm=model, chain_type=chain_type, retriever=retriever,
            condense_question_prompt=condense,
            combine_docs_chain_kwargs={"prompt": answer},
            return_source_documents=True, return_generated_question=True,
        )

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import tempfile
    from pathlib import Path
    import panel as pn
    import param
    
    class cbfs(param.Parameterized):
        chat_history = param.List(default=[])
        answer = param.String(default="")
        db_query = param.String(default="")
        db_response = param.List(default=[])
    
        def __init__(self, default_pdf="docs/cs229_lectures/MachineLearning-Lecture01.pdf", loader=load_db, **params):
            super().__init__(**params)
            self.panels = []
            self.loader = loader
            self.loaded_file = str(default_pdf)
            self.upload_directory = tempfile.TemporaryDirectory(prefix="tdb-chat-upload-")
            self.qa = loader(self.loaded_file, "stuff", 4)
    
        def call_load_db(self, count):
            if count == 0 or not file_input.value:
                return pn.pane.Str(f"Loaded File: {self.loaded_file}")
            # Per-instance storage avoids overwriting another session's temp.pdf.
            upload_path = Path(self.upload_directory.name) / "uploaded.pdf"
            file_input.save(str(upload_path))
            button_load.loading = True
            try:
                new_qa = self.loader(str(upload_path), "stuff", 4)
            finally:
                button_load.loading = False
            self.qa = new_qa
            self.loaded_file = Path(file_input.filename or "uploaded.pdf").name
            self.clr_history()
            return pn.pane.Str(f"Loaded File: {self.loaded_file}")
    
        def convchain(self, query):
            if not query:
                return pn.WidgetBox(*self.panels, scroll=True)
            result = self.qa.invoke({"question": query, "chat_history": self.chat_history})
            # Assign a new list so Param detects the history change.
            self.chat_history = [*self.chat_history, (query, result["answer"])]
            self.db_query = result["generated_question"]
            self.db_response = result["source_documents"]
            self.answer = result["answer"]
            self.panels.extend([
                pn.Row("User:", pn.pane.Str(query, width=600)),
                pn.Row("ChatBot:", pn.pane.Str(self.answer, width=600, styles={"background-color": "#F6F6F6"})),
            ])
            inp.value = ""
            return pn.WidgetBox(*self.panels, scroll=True)
    
        @param.depends("db_query")
        def get_lquest(self):
            return pn.Column(pn.pane.Markdown("**Last question to DB:**"), pn.pane.Str(self.db_query or "No DB accesses so far"))
    
        @param.depends("db_response")
        def get_sources(self):
            return pn.WidgetBox(pn.pane.Markdown("**Result of DB lookup:**"), *[pn.pane.Str(doc) for doc in self.db_response], width=600, scroll=True)
    
        @param.depends("chat_history")
        def get_chats(self):
            return pn.WidgetBox(pn.pane.Markdown("**Current Chat History:**"), *[pn.pane.Str(exchange) for exchange in self.chat_history], width=600, scroll=True)
    
        def clr_history(self, event=None):
            self.chat_history = []
            self.panels = []
            self.answer = ""
            self.db_query = ""
            self.db_response = []
            inp.value = ""
            conversation_view.objects = []
    
    cb = cbfs()
    file_input = pn.widgets.FileInput(accept=".pdf")
    button_load = pn.widgets.Button(name="Load DB", button_type="primary")
    button_clearhistory = pn.widgets.Button(name="Clear History", button_type="warning")
    button_clearhistory.on_click(cb.clr_history)
    inp = pn.widgets.TextInput(placeholder="Enter text here…")
    conversation_view = pn.Column(height=300, scroll=True)
    
    def submit_query(event):
        if event.new:
            conversation_view.objects = [cb.convchain(event.new)]
    
    inp.param.watch(submit_query, "value")
    bound_button_load = pn.bind(cb.call_load_db, button_load.param.clicks)
    tab1 = pn.Column(inp, pn.layout.Divider(), conversation_view)
    tab2 = pn.Column(pn.panel(cb.get_lquest), pn.layout.Divider(), pn.panel(cb.get_sources))
    tab3 = pn.Column(pn.panel(cb.get_chats))
    tab4 = pn.Column(
        pn.Row(file_input, button_load, bound_button_load),
        button_clearhistory,
        pn.pane.Markdown("Upload a text-based PDF. Clearing history starts a new topic. Documents are sent to the configured model in hosted mode."),
    )
    dashboard = pn.Column(
        pn.pane.Markdown("# ChatWithYourData_Bot"),
        pn.Tabs(("Conversation", tab1), ("Database", tab2), ("Chat History", tab3), ("Configure", tab4)),
    )
    print(dashboard)